In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
PROJECT_ROOT = r"D:\Projects\fyp-2"   # 🔴 adjust only if needed
os.chdir(PROJECT_ROOT)

print("Working directory:", os.getcwd())


Working directory: D:\Projects\fyp-2


In [3]:
assert os.path.exists("data/splits/patient_splits.json")
print("Split file found ✔️")


Split file found ✔️


In [4]:
from src.models.attention_fusion import AttentionFusionModel
from src.models.attention_dataset import AttentionFusionDataset
from src.models.collate import collate_attention


In [5]:
train_dataset = AttentionFusionDataset(
    wavlm_root="data/features/wavlm",
    roberta_root="data/features/roberta",
    split_file="data/splits/patient_splits.json",
    split="train",
    label_map={"dementia": 1, "no_dementia": 0}
)

val_dataset = AttentionFusionDataset(
    wavlm_root="data/features/wavlm",
    roberta_root="data/features/roberta",
    split_file="data/splits/patient_splits.json",
    split="val",
    label_map={"dementia": 1, "no_dementia": 0}
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 249
Val samples: 54


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,              # 🔴 keep small (sequence data)
    shuffle=True,
    collate_fn=collate_attention
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_attention
)


In [7]:
model = AttentionFusionModel(
    dim=768,
    heads=4
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model


AttentionFusionModel(
  (audio_proj): Linear(in_features=768, out_features=768, bias=True)
  (text_proj): Linear(in_features=768, out_features=768, bias=True)
  (cross_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)

In [8]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for audio, text, y in loader:
            audio = audio.to(DEVICE)
            text = text.to(DEVICE)
            y = y.to(DEVICE).float()

            logits = model(audio, text)
            probs = torch.sigmoid(logits)

            pred = (probs > 0.5).long()
            preds.extend(pred.cpu().numpy())
            labels.extend(y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)

    return acc, f1, preds, labels


In [9]:
EPOCHS = 20

history = {
    "train_loss": [],
    "val_acc": [],
    "val_f1": []
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for audio, text, y in train_loader:
        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        y = y.to(DEVICE).float()

        optimizer.zero_grad()
        logits = model(audio, text)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    val_acc, val_f1, _, _ = evaluate(model, val_loader)

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.3f} | "
        f"Val F1: {val_f1:.3f}"
    )


Epoch 01 | Train Loss: 0.6709 | Val Acc: 0.574 | Val F1: 0.000
Epoch 02 | Train Loss: 0.6610 | Val Acc: 0.574 | Val F1: 0.000
Epoch 03 | Train Loss: 0.6609 | Val Acc: 0.574 | Val F1: 0.000
Epoch 04 | Train Loss: 0.6562 | Val Acc: 0.574 | Val F1: 0.000
Epoch 05 | Train Loss: 0.6452 | Val Acc: 0.593 | Val F1: 0.083
Epoch 06 | Train Loss: 0.6393 | Val Acc: 0.611 | Val F1: 0.160
Epoch 07 | Train Loss: 0.6083 | Val Acc: 0.574 | Val F1: 0.000
Epoch 08 | Train Loss: 0.6039 | Val Acc: 0.611 | Val F1: 0.160
Epoch 09 | Train Loss: 0.5719 | Val Acc: 0.704 | Val F1: 0.529
Epoch 10 | Train Loss: 0.5677 | Val Acc: 0.611 | Val F1: 0.222
Epoch 11 | Train Loss: 0.5034 | Val Acc: 0.611 | Val F1: 0.160
Epoch 12 | Train Loss: 0.4584 | Val Acc: 0.704 | Val F1: 0.680
Epoch 13 | Train Loss: 0.4626 | Val Acc: 0.704 | Val F1: 0.500
Epoch 14 | Train Loss: 0.4581 | Val Acc: 0.704 | Val F1: 0.500
Epoch 15 | Train Loss: 0.4009 | Val Acc: 0.778 | Val F1: 0.739
Epoch 16 | Train Loss: 0.3678 | Val Acc: 0.796 | Val F1

In [10]:
import numpy as np

best_epoch = np.argmax(history["val_f1"])
print("Best Epoch:", best_epoch + 1)
print("Best Val Accuracy:", history["val_acc"][best_epoch])
print("Best Val F1:", history["val_f1"][best_epoch])


Best Epoch: 16
Best Val Accuracy: 0.7962962962962963
Best Val F1: 0.7555555555555555


In [11]:
_, _, preds, labels = evaluate(model, val_loader)

print(
    classification_report(
        labels,
        preds,
        target_names=["No Dementia", "Dementia"]
    )
)


              precision    recall  f1-score   support

 No Dementia       0.66      0.94      0.77        31
    Dementia       0.80      0.35      0.48        23

    accuracy                           0.69        54
   macro avg       0.73      0.64      0.63        54
weighted avg       0.72      0.69      0.65        54



In [12]:
torch.save(model.state_dict(), "attention_fusion_model.pt")
print("Attention fusion model saved ✔️")


Attention fusion model saved ✔️
